now for car type only

In [39]:
import numpy as np
import os

def dynamics_diversity(M_query, M_ref):
    """
    Measure 'diversity' between two complex dynamics matrices.

    Here we use the angle between them in the flattened complex vector space:
        diversity = arccos( |<v1, v2>| / (||v1|| * ||v2||) )

    Smaller value => more similar.
    """
    v1 = M_query.flatten()
    v2 = M_ref.flatten()

    num = np.vdot(v1, v2)  # complex inner product
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom < 1e-12:
        return np.pi / 2  # treat as maximally diverse

    cos_theta = np.clip(np.abs(num) / denom, 0.0, 1.0)
    theta = np.arccos(cos_theta)
    return float(theta)


In [40]:
def generate_query_dynamics(M_list, base_idx=None, noise_level=0.05):
    """
    Generate a new dynamics matrix M_query by perturbing an existing one.

    This keeps it 'near' something in the DB (more realistic than pure random).
    """
    if base_idx is None:
        base_idx = np.random.randint(len(M_list))
    M_base = M_list[base_idx]
    noise = noise_level * (np.random.randn(*M_base.shape)) ### want this to be real
    M_query = M_base + noise
    return M_query, base_idx


def generate_query_obstacle(grid_H=5, grid_W=5):
    """
    Generate a query obstacle as a car in an interior cell.

    Returns:
        grid_pos : (row, col)
        center   : np.array([cx, cy]) with cx,cy = cell center
    """
    row = np.random.randint(-2*grid_H, 2*grid_H)
    col = np.random.randint(-2*grid_H, 2*grid_H)
    center = np.array([row + 0.5, col + 0.5])
    return (row, col), center


In [41]:
def select_best_cluster_for_query(M_query, db):
    """
    Among all consensus dynamics (one per cluster), find the cluster whose CD
    is closest to M_query in diversity.
    """
    best_cluster_id = None
    best_div = np.inf

    for cluster_id, cd_node in db.consensus_nodes.items():
        M_cd = cd_node.M_cd
        div = dynamics_diversity(M_query, M_cd)
        if div < best_div:
            best_div = div
            best_cluster_id = cluster_id

    return best_cluster_id, best_div


def select_best_dyn_in_cluster(M_query, db, cluster_id):
    """
    Within a given cluster, find the dyn_id whose M is closest to M_query.
    """
    cd_node = db.consensus_nodes[cluster_id]

    best_dyn_id = None
    best_div = np.inf

    for dyn_id, dyn_node in cd_node.dyn_children.items():
        M_dyn = dyn_node.M
        div = dynamics_diversity(M_query, M_dyn)
        if div < best_div:
            best_div = div
            best_dyn_id = dyn_id

    return best_dyn_id, best_div


In [42]:
def obstacle_center_from_grid_pos(grid_pos):
    r, c = grid_pos
    return np.array([r + 0.5, c + 0.5])


def select_closest_car_obstacle(query_center, db, dyn_id):
    """
    For a given dyn_id, look at its 'car' obstacles and find the instance whose
    center is closest (Euclidean) to query_center.
    """
    dyn_node = db.dyn_nodes[dyn_id]

    if "car" not in dyn_node.obs_types:
        raise ValueError(f"Dyn {dyn_id} has no 'car' obstacle type in DB.")

    obs_type = dyn_node.obs_types["car"]

    best_idx = None
    best_dist = np.inf
    best_center = None

    for idx, inst in enumerate(obs_type.instances):
        center = obstacle_center_from_grid_pos(inst.grid_pos)
        dist = np.linalg.norm(center - query_center)
        if dist < best_dist:
            best_dist = dist
            best_idx = idx
            best_center = center

    return best_idx, best_center, best_dist


In [43]:
def load_trajectory_from_db(dyn_id,
                            obs_type_name,
                            instance_idx,
                            traj_dir="traj_output_single"):
    """
    Load a stored trajectory file:
      traj_output_single/traj_dyn_<dyn_id>_<obs_type_name>_<instance_idx>.npz
    """
    fname = f"traj_dyn_{dyn_id}_{obs_type_name}_{instance_idx}.npz"
    path = os.path.join(traj_dir, fname)

    if not os.path.exists(path):
        raise FileNotFoundError(f"Trajectory file not found: {path}")

    data = np.load(path)
    return data  # contains states, inputs, cost, dt, n_horizon, ...


In [44]:
def system1_solve_one_problem(db,
                              M_list,
                              grid_H=5,
                              grid_W=5,
                              traj_dir="traj_output_single",
                              noise_level=0.05):
    """
    1) Generate a problem:
         - query dynamics M_query
         - query car obstacle at some interior grid cell

    2) Similarity search:
         - Find closest consensus dynamics cluster (smallest diversity)
         - Within that cluster, find closest dyn_id
         - For that dyn_id's 'car' obstacles, find closest obstacle position

    3) Retrieve corresponding stored trajectory from disk.

    Returns a dict with everything.
    """

    # ---- 1. Generate problem (dynamics + obstacle) ----
    M_query, base_idx = generate_query_dynamics(M_list, base_idx=None, noise_level=noise_level)
    q_grid_pos, q_center = generate_query_obstacle(grid_H=grid_H, grid_W=grid_W)

    print("=== Generated Problem ===")
    print(f"Query dynamics = {M_query}")
    print(f"Query dynamics built by perturbing M_list[{base_idx}]")
    print(f"Query obstacle grid pos = {q_grid_pos}, center = {q_center}\n")

    # ---- 2a. Best cluster via consensus dynamics ----
    best_cluster_id, div_cluster = select_best_cluster_for_query(M_query, db)
    print(f"Closest cluster ID = {best_cluster_id}, diversity to CD = {div_cluster:.4f}")

    # ---- 2b. Best dyn within that cluster ----
    best_dyn_id, div_dyn = select_best_dyn_in_cluster(M_query, db, best_cluster_id)
    print(f"Closest dyn_id within cluster = {best_dyn_id}, diversity = {div_dyn:.4f}")

    # ---- 2c. Best car obstacle position in that dyn ----
    best_instance_idx, best_center, best_dist = select_closest_car_obstacle(q_center, db, best_dyn_id)
    print(f"Closest 'car' obstacle instance_idx = {best_instance_idx}")
    print(f"  Stored center = {best_center}, distance = {best_dist:.4f}\n")

    # ---- 3. Retrieve trajectory ----
    traj_data = load_trajectory_from_db(
        dyn_id=best_dyn_id,
        obs_type_name="car",
        instance_idx=best_instance_idx,
        traj_dir=traj_dir
    )

    print("Retrieved trajectory info:")
    print(f"  dyn_id      = {traj_data['dyn_id']}")
    print(f"  obs_type    = {traj_data['obs_type']}")
    print(f"  instance    = {traj_data['instance_idx']}")
    print(f"  cost        = {float(traj_data['cost']):.4f}")
    print(f"  dt          = {float(traj_data['dt']):.4f}")
    print(f"  n_horizon   = {int(traj_data['n_horizon'])}")
    print(f"  states shape= {traj_data['states'].shape}")
    print(f"  inputs shape= {traj_data['inputs'].shape}")

    # Bundle everything in a dict for further use
    result = {
        "M_query": M_query,
        "query_obstacle_grid_pos": q_grid_pos,
        "query_obstacle_center": q_center,
        "best_cluster_id": best_cluster_id,
        "best_dyn_id": best_dyn_id,
        "best_dyn_diversity": div_dyn,
        "best_instance_idx": best_instance_idx,
        "best_obstacle_center": best_center,
        "best_obstacle_distance": best_dist,
        "trajectory": traj_data
    }

    return result


db structure (the same as those in S1_layers)

In [45]:
import random
from dataclasses import dataclass, field
from typing import List, Tuple, Dict
import numpy as np

# ---- Node types (same as before, unchanged) --------------------------------

@dataclass
class ObsInstance:
    grid_pos: Tuple[int, int]      # (row, col): 0,0 = upper-left
    timestamp: float = 0.0
    traj_ref: int = None           # can later hold trajectory index/id

@dataclass
class ObsType:
    name: str
    dyn_id: int
    instances: List[ObsInstance] = field(default_factory=list)

@dataclass
class Dyn:
    dyn_id: int
    cluster_id: int
    M: np.ndarray
    obs_types: Dict[str, ObsType] = field(default_factory=dict)

@dataclass
class ConsensusDyn:
    cluster_id: int
    cd_dyn_idx: int
    M_cd: np.ndarray
    dyn_children: Dict[int, Dyn] = field(default_factory=dict)

@dataclass
class HierarchicalDB:
    consensus_nodes: Dict[int, ConsensusDyn] = field(default_factory=dict)
    dyn_to_cluster: Dict[int, int] = field(default_factory=dict)
    dyn_nodes: Dict[int, Dyn] = field(default_factory=dict)


paste the variables M_list and db

In [52]:
import pickle

# Load M_list
with open("M_list.pkl", "rb") as f:
    M_list = pickle.load(f)

# Load db
with open("db.pkl", "rb") as f:
    db = pickle.load(f)

print("Loaded M_list and db!")


Loaded M_list and db!


In [53]:
res = system1_solve_one_problem(
    db=db,
    M_list=M_list,
    grid_H=5,
    grid_W=5,
    traj_dir="traj_output_single",
    noise_level=0.05
)


=== Generated Problem ===
Query dynamics = [[-0.28145566  0.06555046]
 [-0.0453161  -0.30250244]]
Query dynamics built by perturbing M_list[5]
Query obstacle grid pos = (-5, 7), center = [-4.5  7.5]

Closest cluster ID = 1, diversity to CD = 0.1262
Closest dyn_id within cluster = 5, diversity = 0.1262
Closest 'car' obstacle instance_idx = 14
  Stored center = [-4.5         8.83333333], distance = 1.3333



FileNotFoundError: Trajectory file not found: traj_output_single/traj_dyn_5_car_14.npz